# IPL Match Data Analysis
### PBL Activity 4 — Case Study Analysis

**Course:** Python for Data Science (BE05000231)
**Semester:** 5 | **Branch:** Computer Engineering — Artificial Intelligence & Data Science
**Term:** ODD 2026
**Course Outcomes Demonstrated:** CO2, CO4, CO5

## 2. Student Information

| Field | Detail |
|---|---|
| Name | *(fill in)* |
| Enrollment No. | *(fill in)* |
| Branch | Computer Engineering — AI & Data Science |
| Semester | 5 |
| College | *(fill in)* |
| Academic Term | ODD 2026 |

## 3. PBL Problem Statement

Analyze historical Indian Premier League (IPL) match data to answer practical, data-driven questions:

1. Which team performs best overall?
2. Who is the highest run scorer?
3. Who is the most successful bowler?
4. How does performance vary by venue?
5. What is the empirical probability of winning under different conditions (toss result, batting order, venue, etc.)?

All conclusions in this notebook are computed directly from the dataset loaded in Section 8 — no numbers are assumed or fabricated in advance.

## 4. Objectives

- Apply Pandas and NumPy to load, clean, and analyze structured IPL data (**CO2**)
- Demonstrate data cleaning and preparation techniques on a real-world, imperfect dataset (**CO4**)
- Apply Matplotlib and Seaborn to visualize and interpret team, player, venue, and probability trends (**CO5**)
- Build a consolidated analytics dashboard summarizing key findings
- Draw evidence-based conclusions, explicitly stating assumptions and limitations

## 5. Tools and Technologies

| Tool | Purpose |
|---|---|
| Python 3.x | Programming language |
| Pandas | Data loading, cleaning, aggregation |
| NumPy | Numerical and array-based calculations |
| Matplotlib | Core static visualizations |
| Seaborn | Statistical / categorical visualizations |
| Jupyter Notebook | Development and reporting environment |

## 6. Dataset Description

**Source:** Public IPL dataset (Kaggle) distributed as two related CSV files.

**`matches.csv`** — one row per match:
`id, season, city, date, team1, team2, toss_winner, toss_decision, result, winner, win_by_runs, win_by_wickets, player_of_match, venue, umpire1, umpire2`

**`deliveries.csv`** — one row per ball bowled:
`match_id, inning, batting_team, bowling_team, over, ball, batter, bowler, non_striker, batsman_runs, extra_runs, total_runs, is_wicket, dismissal_kind, player_dismissed`

**Relationship:** `deliveries.match_id` is a foreign key referencing `matches.id`. Aggregating `deliveries` by `match_id` (and by player) and merging with `matches` lets us connect ball-level batting/bowling detail to match-level context (season, venue, winner).

> **Note:** Place the two CSV files inside the `data/` folder before running this notebook. The exact season coverage, match count, and other dataset characteristics are **not stated here** — they are computed and printed in Section 9, directly from the files you provide, so nothing below is a guess.

**Do not fabricate:** every statistic that follows this section is computed live from the loaded DataFrames.

## 7. Import Libraries

In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display / plotting configuration
pd.set_option('display.max_columns', 50)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

print("Libraries imported successfully.")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Libraries imported successfully.
Pandas version: 3.0.3
NumPy version: 2.5.1


## 8. Load Dataset

We load both CSV files. If a file is missing, a clear error is raised rather than the notebook silently continuing with no data (a common data-pipeline mistake to avoid).

In [44]:
import os

DATA_DIR = "../data"
matches_path = os.path.join(DATA_DIR, "/home/Leo/Documents/coding/ai/one_way/pds submition/data/matches.csv")
deliveries_path = os.path.join(DATA_DIR, "/home/Leo/Documents/coding/ai/one_way/pds submition/data/deliveries.csv")

if not os.path.exists(matches_path) or not os.path.exists(deliveries_path):
    raise FileNotFoundError(
        f"Expected 'matches.csv' and 'deliveries.csv' inside {DATA_DIR}/. "
        "Download the IPL dataset from Kaggle and place both files there before re-running."
    )

matches = pd.read_csv(matches_path)
deliveries = pd.read_csv(deliveries_path)

print("matches.csv loaded:", matches.shape)
print("deliveries.csv loaded:", deliveries.shape)

matches.csv loaded: (1095, 20)
deliveries.csv loaded: (260920, 17)


## 9. Data Understanding

Standard first-look diagnostics (as per lab manual Practical–13, Program 12) applied to both files.

In [45]:
print("===== MATCHES: head =====")
display(matches.head())

print("\n===== MATCHES: shape =====")
print(matches.shape)

print("\n===== MATCHES: info =====")
matches.info()

print("\n===== MATCHES: describe (numeric) =====")
display(matches.describe())

===== MATCHES: head =====


,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan



===== MATCHES: shape =====
(1095, 20)

===== MATCHES: info =====
<class 'pandas.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               1095 non-null   int64  
 1   season           1095 non-null   str    
 2   city             1044 non-null   str    
 3   date             1095 non-null   str    
 4   match_type       1095 non-null   str    
 5   player_of_match  1090 non-null   str    
 6   venue            1095 non-null   str    
 7   team1            1095 non-null   str    
 8   team2            1095 non-null   str    
 9   toss_winner      1095 non-null   str    
 10  toss_decision    1095 non-null   str    
 11  winner           1090 non-null   str    
 12  result           1095 non-null   str    
 13  result_margin    1076 non-null   float64
 14  target_runs      1092 non-null   float64
 15  target_overs     1092 non-null   float64
 16  super

,id,result_margin,target_runs,target_overs
count,1.095000e+03,1076.000000,1092.000000,1092.000000
mean,9.048283e+05,17.259294,165.684066,19.759341
std,3.677402e+05,21.787444,33.427048,1.581108
min,3.359820e+05,1.000000,43.000000,5.000000
25%,5.483315e+05,6.000000,146.000000,20.000000
50%,9.809610e+05,8.000000,166.000000,20.000000
75%,1.254062e+06,20.000000,187.000000,20.000000
max,1.426312e+06,146.000000,288.000000,20.000000


In [46]:
print("===== MATCHES: missing values =====")
print(matches.isnull().sum())

print("\n===== MATCHES: duplicate rows =====")
print("Duplicates =", matches.duplicated().sum())

===== MATCHES: missing values =====
id                    0
season                0
city                 51
date                  0
match_type            0
player_of_match       5
venue                 0
team1                 0
team2                 0
toss_winner           0
toss_decision         0
winner                5
result                0
result_margin        19
target_runs           3
target_overs          3
super_over            0
method             1074
umpire1               0
umpire2               0
dtype: int64

===== MATCHES: duplicate rows =====
Duplicates = 0


In [47]:
print("===== DELIVERIES: head =====")
display(deliveries.head())

print("\n===== DELIVERIES: shape =====")
print(deliveries.shape)

print("\n===== DELIVERIES: info =====")
deliveries.info()

===== DELIVERIES: head =====


,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,NaN,NaN,NaN
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,NaN,NaN,NaN
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN



===== DELIVERIES: shape =====
(260920, 17)

===== DELIVERIES: info =====
<class 'pandas.DataFrame'>
RangeIndex: 260920 entries, 0 to 260919
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   match_id          260920 non-null  int64
 1   inning            260920 non-null  int64
 2   batting_team      260920 non-null  str  
 3   bowling_team      260920 non-null  str  
 4   over              260920 non-null  int64
 5   ball              260920 non-null  int64
 6   batter            260920 non-null  str  
 7   bowler            260920 non-null  str  
 8   non_striker       260920 non-null  str  
 9   batsman_runs      260920 non-null  int64
 10  extra_runs        260920 non-null  int64
 11  total_runs        260920 non-null  int64
 12  extras_type       14125 non-null   str  
 13  is_wicket         260920 non-null  int64
 14  player_dismissed  12950 non-null   str  
 15  dismissal_kind    12950 non-null   str 

In [48]:
print("===== DELIVERIES: missing values =====")
print(deliveries.isnull().sum())

print("\n===== DELIVERIES: duplicate rows =====")
print("Duplicates =", deliveries.duplicated().sum())

===== DELIVERIES: missing values =====
match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batter                   0
bowler                   0
non_striker              0
batsman_runs             0
extra_runs               0
total_runs               0
extras_type         246795
is_wicket                0
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64

===== DELIVERIES: duplicate rows =====
Duplicates = 0


In [49]:
# Real dataset coverage — computed, not assumed
n_seasons = matches['season'].nunique()
n_matches = matches['id'].nunique()
n_teams = pd.unique(matches[['team1', 'team2']].values.ravel())
n_venues = matches['venue'].nunique()
date_min, date_max = matches['date'].min(), matches['date'].max()

print(f"Number of seasons        : {n_seasons}")
print(f"Number of matches        : {n_matches}")
print(f"Number of distinct teams : {len(n_teams)}")
print(f"Number of venues         : {n_venues}")
print(f"Date range                : {date_min} to {date_max}")
print(f"Number of delivery rows  : {deliveries.shape[0]}")

Number of seasons        : 17
Number of matches        : 1095
Number of distinct teams : 19
Number of venues         : 58
Date range                : 2008-04-18 to 2024-05-26
Number of delivery rows  : 260920


**Interpretation:** The printed counts above are the authoritative dataset scope for this project. If any of these numbers looks unusually low (e.g., very few seasons), it may indicate the CSV downloaded is a partial/sample version — re-check the Kaggle source before proceeding with analysis.

## 10. Data Cleaning

Following the lab manual's cleaning workflow (Practical–14): handle missing values, duplicates, and inconsistent categorical labels — team name changes are the most common IPL data-quality issue (e.g. a franchise rebranding across seasons).

In [50]:
# 10.1 Standardize inconsistent team names
# IPL franchises have been renamed/rebranded over the years. We map old names
# to their current identity so the same franchise isn't double-counted as two teams.
team_name_map = {
    "Delhi Daredevils": "Delhi Capitals",
    "Deccan Chargers": "Sunrisers Hyderabad",
    "Kings XI Punjab": "Punjab Kings",
    "Rising Pune Supergiant": "Rising Pune Supergiants",
}

for col in ["team1", "team2", "toss_winner", "winner"]:
    if col in matches.columns:
        matches[col] = matches[col].replace(team_name_map)

for col in ["batting_team", "bowling_team"]:
    if col in deliveries.columns:
        deliveries[col] = deliveries[col].replace(team_name_map)

print("Team names standardized. Distinct teams now:",
      len(pd.unique(matches[['team1', 'team2']].values.ravel())))

Team names standardized. Distinct teams now: 15


In [51]:
# 10.2 Handle missing / no-result matches
# A match can have no winner if it was abandoned or tied with no super over recorded.
missing_winner = matches['winner'].isnull().sum()
print("Matches with no recorded winner:", missing_winner)

# We keep these rows for match-count context but exclude them from win-based
# calculations later using .dropna(subset=['winner']) at the point of use,
# rather than deleting them from the dataset entirely (they are still valid
# "matches played" for participation counts).
matches['winner'] = matches['winner'].fillna("No Result")

Matches with no recorded winner: 5


In [52]:
# 10.3 Remove duplicate rows, if any
before = matches.shape[0]
matches = matches.drop_duplicates()
after = matches.shape[0]
print(f"Matches: removed {before - after} duplicate rows.")

before = deliveries.shape[0]
deliveries = deliveries.drop_duplicates()
after = deliveries.shape[0]
print(f"Deliveries: removed {before - after} duplicate rows.")

Matches: removed 0 duplicate rows.
Deliveries: removed 0 duplicate rows.


In [53]:
# 10.4 Standardize venue names (strip whitespace / casing inconsistencies)
matches['venue'] = matches['venue'].astype(str).str.strip()

# Some datasets record the same ground under slightly different names
# (e.g. with/without city suffix). We only trim whitespace here rather than
# forcing a fuzzy-merge, since an incorrect manual merge could misattribute
# venue statistics — this limitation is noted in Section 23.
print("Distinct venues after cleaning:", matches['venue'].nunique())

Distinct venues after cleaning: 58


In [54]:
# 10.5 Date conversion
matches['date'] = pd.to_datetime(matches['date'], errors='coerce')
print("Date column dtype:", matches['date'].dtype)
print("Rows where date failed to parse:", matches['date'].isnull().sum())

Date column dtype: datetime64[us]
Rows where date failed to parse: 0


## 11. Data Preparation

Merge match-level and ball-level data, and create derived columns needed for later analysis (win margin type, batting-first/chasing flag, etc.).

In [55]:
# 11.1 Merge deliveries with match context (season, venue, winner) via match_id -> id
deliveries_full = deliveries.merge(
    matches[['id', 'season', 'venue', 'winner', 'team1', 'team2']],
    left_on='match_id', right_on='id', how='left'
)
print("Merged deliveries_full shape:", deliveries_full.shape)
deliveries_full.head()

Merged deliveries_full shape: (260920, 23)


,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder,id,season,venue,winner,team1,team2
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,NaN,NaN,NaN,335982,2007/08,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,Kolkata Knight Riders
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN,335982,2007/08,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,Kolkata Knight Riders
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,NaN,NaN,NaN,335982,2007/08,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,Kolkata Knight Riders
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN,335982,2007/08,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,Kolkata Knight Riders
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN,335982,2007/08,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,Kolkata Knight Riders


In [56]:
# 11.2 Derived column: result type of victory
def classify_result(row):
    if row['winner'] == "No Result":
        return "No Result"
    if row.get('win_by_runs', 0) and row['win_by_runs'] > 0:
        return "Won batting first (by runs)"
    if row.get('win_by_wickets', 0) and row['win_by_wickets'] > 0:
        return "Won chasing (by wickets)"
    return "Other/Tie"

matches['victory_type'] = matches.apply(classify_result, axis=1)
matches[['winner', 'win_by_runs', 'win_by_wickets', 'victory_type']].head()

KeyError: "['win_by_runs', 'win_by_wickets'] not in index"

In [ ]:
# 11.3 Derived column: did the toss winner also win the match?
matches['toss_winner_won_match'] = np.where(
    matches['toss_winner'] == matches['winner'], 'Yes',
    np.where(matches['winner'] == 'No Result', 'No Result', 'No')
)
matches['toss_winner_won_match'].value_counts()

In [ ]:
# 11.4 GroupBy example: matches per season
matches_per_season = matches.groupby('season')['id'].count().sort_index()
print(matches_per_season)

## 12. Exploratory Data Analysis (EDA)

Structured per the standard EDA workflow: overview → dimensions → dtypes → missing/duplicate check (already done in Sections 9–10) → categorical → numerical → univariate → bivariate → multivariate → correlation → key observations.

### 12.1 Categorical overview

In [ ]:
categorical_cols = ['season', 'venue', 'toss_decision', 'winner']
for c in categorical_cols:
    print(f"--- {c} : {matches[c].nunique()} unique values ---")
    print(matches[c].value_counts().head(5))
    print()

### 12.2 Univariate analysis — matches per season

In [ ]:
plt.figure(figsize=(10, 5))
matches_per_season.plot(kind='bar', color='steelblue')
plt.title("Number of IPL Matches per Season")
plt.xlabel("Season")
plt.ylabel("Number of Matches")
plt.grid(axis='y')
plt.tight_layout()
plt.savefig("../figures/matches_per_season.png")
plt.show()

**Observation:** The bar chart shows how many matches were played each season.
**Possible reason:** Season-to-season variation typically reflects changes in the number of participating franchises or format (e.g. addition of new teams).
**Analytical implication:** Any team-count-based comparison across seasons should account for this — a season with more matches naturally offers more chances to accumulate wins/runs/wickets.

### 12.3 Univariate analysis — toss decision distribution

In [ ]:
plt.figure(figsize=(6, 6))
matches['toss_decision'].value_counts().plot(
    kind='pie', autopct='%1.1f%%', startangle=90
)
plt.title("Toss Decision Split (Bat vs Field)")
plt.ylabel("")
plt.tight_layout()
plt.savefig("../figures/toss_decision_pie.png")
plt.show()

### 12.4 Bivariate analysis — toss decision vs match outcome

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=matches[matches['winner'] != 'No Result'],
              x='toss_decision', hue='toss_winner_won_match')
plt.title("Toss Decision vs Whether Toss Winner Won the Match")
plt.xlabel("Toss Decision")
plt.ylabel("Number of Matches")
plt.legend(title="Toss winner won?")
plt.tight_layout()
plt.savefig("../figures/toss_decision_vs_outcome.png")
plt.show()

### 12.5 Numerical distribution — total runs per match

In [ ]:
runs_per_match = deliveries.groupby('match_id')['total_runs'].sum()

plt.figure(figsize=(9, 5))
plt.hist(runs_per_match, bins=20, color='darkorange', edgecolor='black')
plt.title("Distribution of Total Runs per Match")
plt.xlabel("Total Runs in a Match")
plt.ylabel("Number of Matches")
plt.grid(axis='y')
plt.tight_layout()
plt.savefig("../figures/runs_per_match_hist.png")
plt.show()

print("Mean runs/match  :", np.round(runs_per_match.mean(), 2))
print("Median runs/match:", np.round(runs_per_match.median(), 2))
print("Std deviation    :", np.round(runs_per_match.std(), 2))

### 12.6 Multivariate / correlation analysis

In [ ]:
numeric_match_cols = matches.select_dtypes(include='number')
plt.figure(figsize=(8, 6))
sns.heatmap(numeric_match_cols.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap — Numeric Match Columns")
plt.tight_layout()
plt.savefig("../figures/correlation_heatmap.png")
plt.show()

**Observation:** The heatmap shows pairwise correlation among numeric match columns (e.g. `win_by_runs`, `win_by_wickets`).
**Possible reason:** These two columns are structurally near-mutually-exclusive (a match is won by either runs or wickets, not both), which explains any strong negative association.
**Analytical implication:** Correlation here reflects the scoring rules of cricket, not a causal relationship — this is flagged explicitly to avoid overinterpretation (see Section 23, Limitations).

## 13. Team Performance Analysis

**How "best team" is defined in this project:** raw win count alone is misleading because teams that played more matches (older/more consistent franchises) naturally accumulate more wins. We instead compute, for every team:

- Matches played
- Wins
- Losses
- **Win percentage** = wins / matches played × 100

and rank primarily by win percentage (with a minimum matches-played threshold to avoid a team with 2 matches and 2 wins looking "best"), rather than by total wins alone.

In [ ]:
def team_match_counts(matches_df):
    t1 = matches_df['team1'].value_counts()
    t2 = matches_df['team2'].value_counts()
    played = t1.add(t2, fill_value=0)
    return played

matches_played = team_match_counts(matches)
wins = matches[matches['winner'] != 'No Result']['winner'].value_counts()

team_stats = pd.DataFrame({
    'matches_played': matches_played,
    'wins': wins
}).fillna(0)

team_stats['losses'] = team_stats['matches_played'] - team_stats['wins']
team_stats['win_pct'] = np.round((team_stats['wins'] / team_stats['matches_played']) * 100, 2)
team_stats = team_stats.sort_values('win_pct', ascending=False)
team_stats.head(10)

In [ ]:
# Apply a minimum-matches threshold before declaring a "best team" by win%,
# so a team with very few matches doesn't distort the ranking.
MIN_MATCHES = 20
qualified_teams = team_stats[team_stats['matches_played'] >= MIN_MATCHES]
best_team_by_pct = qualified_teams.sort_values('win_pct', ascending=False).head(10)
print(f"Teams with at least {MIN_MATCHES} matches, ranked by win %:")
best_team_by_pct

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

team_stats.sort_values('wins', ascending=False).head(10)['wins'].plot(
    kind='bar', ax=axes[0], color='seagreen'
)
axes[0].set_title("Top 10 Teams by Total Wins")
axes[0].set_ylabel("Wins")
axes[0].set_xlabel("Team")
axes[0].tick_params(axis='x', rotation=75)

best_team_by_pct['win_pct'].plot(kind='bar', ax=axes[1], color='indianred')
axes[1].set_title(f"Top 10 Teams by Win % (min {MIN_MATCHES} matches)")
axes[1].set_ylabel("Win Percentage")
axes[1].set_xlabel("Team")
axes[1].tick_params(axis='x', rotation=75)

plt.tight_layout()
plt.savefig("../figures/team_performance.png")
plt.show()

**Observation:** Compare the two charts — the "most wins" leader is not always the "highest win %" leader.
**Possible reason:** A franchise that has played in every season accumulates more total wins simply through more opportunities.
**Analytical implication:** For this project, the team with the **highest win percentage (subject to the minimum-matches threshold)** is treated as the best-performing team, since it best isolates skill/consistency from longevity.

In [ ]:
# Season-wise win trend for the current win%-leading team (dynamically determined, not hardcoded)
top_team = best_team_by_pct.index[0]
team_season_wins = matches[matches['winner'] == top_team].groupby('season')['id'].count()

plt.figure(figsize=(10, 5))
team_season_wins.plot(kind='line', marker='o', color='purple')
plt.title(f"Season-wise Wins — {top_team}")
plt.xlabel("Season")
plt.ylabel("Wins")
plt.grid(True)
plt.tight_layout()
plt.savefig("../figures/top_team_season_trend.png")
plt.show()

## 14. Player Batting Analysis

**Metric definition:** total runs is the primary ranking metric, but we explicitly also report matches/innings played alongside it, since a batsman with more innings has a structural advantage in a cumulative-total metric. We report **runs per match** as a secondary, more workload-normalized metric.

In [ ]:
batting = deliveries.groupby('batter').agg(
    total_runs=('batsman_runs', 'sum'),
    balls_faced=('batsman_runs', 'count'),
    matches_played=('match_id', 'nunique')
).reset_index()

batting['runs_per_match'] = np.round(batting['total_runs'] / batting['matches_played'], 2)
batting['strike_rate'] = np.round((batting['total_runs'] / batting['balls_faced']) * 100, 2)

top_run_scorers = batting.sort_values('total_runs', ascending=False).head(10)
top_run_scorers

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(top_run_scorers['batter'], top_run_scorers['total_runs'], color='teal')
plt.title("Top 10 Run Scorers (All Seasons Combined)")
plt.xlabel("Total Runs")
plt.ylabel("Batsman")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("../figures/top_batsmen.png")
plt.show()

In [ ]:
# Bias check: does the run-scoring leader also have a high matches_played count?
plt.figure(figsize=(8, 6))
plt.scatter(batting['matches_played'], batting['total_runs'], alpha=0.4, color='gray')
plt.scatter(top_run_scorers['matches_played'], top_run_scorers['total_runs'], color='red', label='Top 10 scorers')
plt.title("Matches Played vs Total Runs (All Batsmen)")
plt.xlabel("Matches Played")
plt.ylabel("Total Runs")
plt.legend()
plt.tight_layout()
plt.savefig("../figures/runs_vs_matches_scatter.png")
plt.show()

**Observation:** The scatter plot shows whether the leading run-scorers are simply the players who have played the most matches.
**Limitation stated explicitly:** Total runs rewards longevity as much as skill. `runs_per_match` (computed above) is provided as a fairer, workload-adjusted comparison — see the table for how rankings shift under that metric.

In [ ]:
# Runs-per-match leaderboard (minimum matches threshold to avoid 1-match outliers)
MIN_MATCHES_BAT = 10
qualified_batters = batting[batting['matches_played'] >= MIN_MATCHES_BAT]
top_by_rpm = qualified_batters.sort_values('runs_per_match', ascending=False).head(10)
top_by_rpm

In [ ]:
# Season-wise batting trend for the top run scorer (dynamic, not hardcoded)
top_scorer_name = top_run_scorers.iloc[0]['batter']
season_runs = deliveries_full[deliveries_full['batter'] == top_scorer_name].groupby('season')['batsman_runs'].sum()

plt.figure(figsize=(10, 5))
season_runs.plot(kind='line', marker='o', color='darkgreen')
plt.title(f"Season-wise Runs — {top_scorer_name}")
plt.xlabel("Season")
plt.ylabel("Runs")
plt.grid(True)
plt.tight_layout()
plt.savefig("../figures/top_scorer_season_trend.png")
plt.show()

## 15. Bowling Analysis

**Metric definition — "most successful bowler":** total wickets is the headline metric, but economy rate (runs conceded per over) is reported alongside it as a measure of efficiency, since a bowler who took many wickets while conceding many runs is not straightforwardly "better" than a more economical bowler. Both are shown; we do not collapse them into a single arbitrary score without justification.

**Assumption:** a wicket is counted as credited to the bowler whenever `is_wicket == 1`, excluding dismissal kinds not creditable to the bowler (run out, retired hurt, obstructing the field) if that column is present in the dataset.

In [ ]:
non_bowler_dismissals = ['run out', 'retired hurt', 'obstructing the field']

bowler_wickets = deliveries[
    (deliveries['is_wicket'] == 1) &
    (~deliveries['dismissal_kind'].isin(non_bowler_dismissals))
].groupby('bowler')['is_wicket'].sum()

bowler_runs_conceded = deliveries.groupby('bowler')['total_runs'].sum()
bowler_balls = deliveries.groupby('bowler')['ball'].count()
bowler_matches = deliveries.groupby('bowler')['match_id'].nunique()

bowling = pd.DataFrame({
    'wickets': bowler_wickets,
    'runs_conceded': bowler_runs_conceded,
    'balls_bowled': bowler_balls,
    'matches_played': bowler_matches
}).fillna(0)

bowling['overs_bowled'] = np.round(bowling['balls_bowled'] / 6, 1)
bowling['economy'] = np.round(bowling['runs_conceded'] / (bowling['balls_bowled'] / 6), 2)
bowling['wickets_per_match'] = np.round(bowling['wickets'] / bowling['matches_played'], 2)

top_wicket_takers = bowling.sort_values('wickets', ascending=False).head(10)
top_wicket_takers

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(top_wicket_takers.index, top_wicket_takers['wickets'], color='crimson')
plt.title("Top 10 Wicket Takers (All Seasons Combined)")
plt.xlabel("Total Wickets")
plt.ylabel("Bowler")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("../figures/top_bowlers.png")
plt.show()

In [ ]:
# Economy rate among the top wicket takers (min balls bowled to avoid tiny-sample distortion)
MIN_BALLS = 120  # 20 overs
qualified_bowlers = bowling[bowling['balls_bowled'] >= MIN_BALLS]

plt.figure(figsize=(9, 5))
sns.boxplot(x=qualified_bowlers['economy'])
plt.title("Economy Rate Distribution (Bowlers with 20+ Overs Bowled)")
plt.xlabel("Economy Rate (runs per over)")
plt.tight_layout()
plt.savefig("../figures/bowling_economy_box.png")
plt.show()

print("Most economical bowler (min 20 overs):",
      qualified_bowlers.sort_values('economy').index[0])

## 16. Venue-wise Performance Analysis

In [ ]:
venue_match_counts = matches['venue'].value_counts().head(10)

plt.figure(figsize=(10, 6))
venue_match_counts.plot(kind='barh', color='slateblue')
plt.title("Top 10 Most Frequently Used Venues")
plt.xlabel("Number of Matches")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("../figures/venue_frequency.png")
plt.show()

In [ ]:
# First-innings score by venue (top 10 most-used venues only, for readability)
first_innings = deliveries_full[deliveries_full['inning'] == 1]
first_innings_scores = first_innings.groupby('match_id')['total_runs'].sum().reset_index()
first_innings_scores = first_innings_scores.merge(matches[['id', 'venue']], left_on='match_id', right_on='id')

top_venues = venue_match_counts.index.tolist()
plot_data = first_innings_scores[first_innings_scores['venue'].isin(top_venues)]

plt.figure(figsize=(12, 6))
sns.boxplot(data=plot_data, x='venue', y='total_runs')
plt.title("First-Innings Score Distribution — Top 10 Venues")
plt.xlabel("Venue")
plt.ylabel("First-Innings Total Runs")
plt.xticks(rotation=80)
plt.tight_layout()
plt.savefig("../figures/venue_score_boxplot.png")
plt.show()

In [ ]:
avg_first_innings_by_venue = plot_data.groupby('venue')['total_runs'].mean().round(2).sort_values(ascending=False)
print("Average first-innings score — top 10 venues:")
print(avg_first_innings_by_venue)

**Observation:** Some venues consistently show higher first-innings totals than others.
**Possible reason:** Ground dimensions, pitch behavior, and altitude are known cricketing factors, though this dataset alone cannot confirm which specific factor drives the difference.
**Analytical implication:** Venue should be treated as a relevant context variable in any pre-match analysis, without claiming a specific physical cause the data doesn't establish.

## 17. Toss Analysis

In [ ]:
toss_decision_counts = matches['toss_decision'].value_counts()
print(toss_decision_counts)

valid_matches = matches[matches['winner'] != 'No Result']
toss_win_rate_valid = (valid_matches['toss_winner'] == valid_matches['winner']).mean() * 100
print(f"\nPercentage of matches where toss winner also won: {np.round(toss_win_rate_valid, 2)}%")

**Interpretation:** A toss-winner win rate close to 50% would suggest the toss has little inherent advantage; a rate meaningfully above 50% would suggest some advantage. The exact figure printed above is this dataset's actual historical rate — read it directly rather than assuming an outcome.

## 18. Winning Probability Analysis

All probabilities below are **empirical (historical frequency-based)**, calculated as *favorable outcomes / total outcomes* from this dataset. This is descriptive statistics, not a predictive machine-learning model — no classifier is trained, and no accuracy score is claimed. Historical frequency does not guarantee a specific future match outcome.

In [ ]:
# Team win probability = wins / matches played
team_stats['win_probability'] = np.round(team_stats['wins'] / team_stats['matches_played'], 3)
team_stats[['matches_played', 'wins', 'win_probability']].sort_values('win_probability', ascending=False).head(10)

In [ ]:
# Batting-first vs chasing win probability (league-wide)
bat_first_wins = (matches['victory_type'] == "Won batting first (by runs)").sum()
chase_wins = (matches['victory_type'] == "Won chasing (by wickets)").sum()
total_decided = bat_first_wins + chase_wins

print(f"Batting-first win probability: {np.round(bat_first_wins / total_decided, 3)}")
print(f"Chasing win probability      : {np.round(chase_wins / total_decided, 3)}")

In [ ]:
plt.figure(figsize=(6, 6))
plt.pie([bat_first_wins, chase_wins], labels=['Batting First Wins', 'Chasing Wins'],
        autopct='%1.1f%%', colors=['gold', 'lightskyblue'])
plt.title("League-wide: Batting First vs Chasing Win Share")
plt.tight_layout()
plt.savefig("../figures/bat_first_vs_chase.png")
plt.show()

In [ ]:
# Toss winner -> match winner probability (recomputed here explicitly as a probability)
print(f"P(toss winner wins match) = {np.round(toss_win_rate_valid / 100, 3)}")

## 19. Advanced IPL Insights

In [ ]:
# 19.1 Average winning margin (runs-based wins only)
avg_margin_runs = matches.loc[matches['win_by_runs'] > 0, 'win_by_runs'].mean()
avg_margin_wkts = matches.loc[matches['win_by_wickets'] > 0, 'win_by_wickets'].mean()
print(f"Average winning margin (by runs)    : {np.round(avg_margin_runs, 2)}")
print(f"Average winning margin (by wickets) : {np.round(avg_margin_wkts, 2)}")

In [ ]:
# 19.2 Most common victory type
print(matches['victory_type'].value_counts())

In [ ]:
# 19.3 Sixes and fours leaders
boundary_counts = deliveries[deliveries['batsman_runs'].isin([4, 6])].groupby(
    ['batter', 'batsman_runs']
).size().unstack(fill_value=0)
boundary_counts.columns = ['fours' if c == 4 else 'sixes' for c in boundary_counts.columns]
boundary_counts['total_boundaries'] = boundary_counts.sum(axis=1)
top_boundary_hitters = boundary_counts.sort_values('total_boundaries', ascending=False).head(10)
top_boundary_hitters

In [ ]:
top_boundary_hitters[['fours', 'sixes']].plot(kind='bar', stacked=True, figsize=(10, 6),
                                                color=['orange', 'purple'])
plt.title("Top 10 Boundary Hitters (Fours + Sixes)")
plt.ylabel("Count")
plt.xlabel("Batsman")
plt.xticks(rotation=75)
plt.tight_layout()
plt.savefig("../figures/boundary_leaders.png")
plt.show()

In [ ]:
# 19.4 Season-wise scoring trend (average total runs per match, by season)
season_totals = deliveries_full.groupby(['season', 'match_id'])['total_runs'].sum().reset_index()
season_avg_runs = season_totals.groupby('season')['total_runs'].mean().round(2)

plt.figure(figsize=(10, 5))
season_avg_runs.plot(kind='line', marker='o', color='brown')
plt.title("Average Total Match Runs by Season")
plt.xlabel("Season")
plt.ylabel("Average Runs per Match")
plt.grid(True)
plt.tight_layout()
plt.savefig("../figures/season_scoring_trend.png")
plt.show()

In [ ]:
# 19.5 Team-vs-team head-to-head matrix (win counts)
head_to_head_wins = matches[matches['winner'] != 'No Result'].pivot_table(
    index='winner', columns='team1', values='id', aggfunc='count', fill_value=0
)
plt.figure(figsize=(12, 8))
sns.heatmap(head_to_head_wins, cmap='YlGnBu', annot=False)
plt.title("Team Win Counts vs Opponent (team1 side) — Head-to-Head Overview")
plt.xlabel("Opponent (as team1)")
plt.ylabel("Winning Team")
plt.tight_layout()
plt.savefig("../figures/head_to_head_heatmap.png")
plt.show()

**Note on this heatmap:** because `team1`/`team2` assignment in raw match data is not always the same as "home/away," this matrix is an overview of win distribution rather than a precise home-ground head-to-head record. This limitation is noted in Section 23.

## 20. Dashboard

A consolidated single-figure dashboard summarizing the KPIs and key charts from all sections above. See `dashboard/IPL_Dashboard.py` for the standalone script version (Phase 3 deliverable) — the same logic is reproduced here for an in-notebook view.

In [ ]:
# KPI values (all computed above, referenced here — nothing new invented)
kpi_total_matches = n_matches
kpi_total_seasons = n_seasons
kpi_total_teams = len(pd.unique(matches[['team1', 'team2']].values.ravel()))
kpi_total_venues = n_venues
kpi_best_team = best_team_by_pct.index[0]
kpi_top_scorer = top_run_scorers.iloc[0]['batter']
kpi_top_wicket_taker = top_wicket_takers.index[0]

print("IPL MATCH ANALYTICS DASHBOARD — KPI SUMMARY")
print("-" * 45)
print(f"Total Matches        : {kpi_total_matches}")
print(f"Total Seasons        : {kpi_total_seasons}")
print(f"Total Teams          : {kpi_total_teams}")
print(f"Total Venues         : {kpi_total_venues}")
print(f"Best Team (win %)    : {kpi_best_team}")
print(f"Highest Run Scorer   : {kpi_top_scorer}")
print(f"Highest Wicket Taker : {kpi_top_wicket_taker}")

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle("IPL MATCH ANALYTICS DASHBOARD", fontsize=18, fontweight='bold')
gs = fig.add_gridspec(3, 3, hspace=0.6, wspace=0.35)

# Panel 1: KPI text summary
ax0 = fig.add_subplot(gs[0, 0])
ax0.axis('off')
kpi_text = (
    f"Total Matches: {kpi_total_matches}\n"
    f"Total Seasons: {kpi_total_seasons}\n"
    f"Total Teams: {kpi_total_teams}\n"
    f"Total Venues: {kpi_total_venues}\n\n"
    f"Best Team: {kpi_best_team}\n"
    f"Top Scorer: {kpi_top_scorer}\n"
    f"Top Wicket-Taker: {kpi_top_wicket_taker}"
)
ax0.text(0, 1, kpi_text, fontsize=11, va='top')
ax0.set_title("Overview KPIs", fontweight='bold')

# Panel 2: Team win %
ax1 = fig.add_subplot(gs[0, 1])
best_team_by_pct['win_pct'].head(5).plot(kind='bar', ax=ax1, color='indianred')
ax1.set_title("Top 5 Teams — Win %")
ax1.tick_params(axis='x', rotation=60)

# Panel 3: Season trend
ax2 = fig.add_subplot(gs[0, 2])
season_avg_runs.plot(kind='line', marker='o', ax=ax2, color='brown')
ax2.set_title("Avg Runs/Match by Season")
ax2.tick_params(axis='x', rotation=60)

# Panel 4: Top batsmen
ax3 = fig.add_subplot(gs[1, 0])
top_run_scorers.set_index('batter')['total_runs'].head(5).plot(kind='barh', ax=ax3, color='teal')
ax3.invert_yaxis()
ax3.set_title("Top 5 Run Scorers")

# Panel 5: Top bowlers
ax4 = fig.add_subplot(gs[1, 1])
top_wicket_takers['wickets'].head(5).plot(kind='barh', ax=ax4, color='crimson')
ax4.invert_yaxis()
ax4.set_title("Top 5 Wicket Takers")

# Panel 6: Venue frequency
ax5 = fig.add_subplot(gs[1, 2])
venue_match_counts.head(5).plot(kind='barh', ax=ax5, color='slateblue')
ax5.invert_yaxis()
ax5.set_title("Top 5 Venues (Matches)")

# Panel 7: Toss decision
ax6 = fig.add_subplot(gs[2, 0])
toss_decision_counts.plot(kind='pie', autopct='%1.1f%%', ax=ax6)
ax6.set_ylabel("")
ax6.set_title("Toss Decision Split")

# Panel 8: Bat first vs chase
ax7 = fig.add_subplot(gs[2, 1])
ax7.pie([bat_first_wins, chase_wins], labels=['Bat First', 'Chase'], autopct='%1.1f%%',
        colors=['gold', 'lightskyblue'])
ax7.set_title("Bat First vs Chase Wins")

# Panel 9: Key insights text
ax8 = fig.add_subplot(gs[2, 2])
ax8.axis('off')
insights_text = (
    f"- Toss winner won match: {np.round(toss_win_rate_valid, 1)}% of matches\n"
    f"- Avg 1st-innings score varies by venue\n"
    f"  (see Section 16)\n"
    f"- Batting-first win share: {np.round(100*bat_first_wins/total_decided,1)}%\n"
    f"- Chasing win share: {np.round(100*chase_wins/total_decided,1)}%"
)
ax8.text(0, 1, insights_text, fontsize=10, va='top')
ax8.set_title("Key Insights", fontweight='bold')

plt.savefig("../figures/dashboard.png", dpi=150, bbox_inches='tight')
plt.show()

## 21. Key Findings

*(Fill in after running this notebook on the real data — write 5–10 bullet points, each citing the actual computed number from the corresponding section above. Do not write findings before running the code.)*

Template (replace the placeholders with the printed values from Sections 13–19):
- Best-performing team by win % (min matches threshold): **`<kpi_best_team>`**
- Highest run scorer: **`<kpi_top_scorer>`** with `<total_runs>` runs
- Most successful bowler by wickets: **`<kpi_top_wicket_taker>`** with `<wickets>` wickets
- Toss winner won the match in `<toss_win_rate_valid>`% of matches
- Batting-first win probability vs chasing win probability: `<...>` vs `<...>`
- Highest average first-innings scoring venue: `<...>`
- (add further findings as observed)

## 22. Conclusion

This project applied Pandas, NumPy, Matplotlib, and Seaborn to a real-world IPL dataset to answer five practical cricket-analytics questions. Team performance was assessed using win percentage (not raw wins) to fairly compare franchises with different tenures. Batting and bowling leaders were identified with explicit attention to the bias introduced by matches played. Venue and toss effects were quantified using empirical probabilities rather than assumed intuitions. All figures and conclusions in this notebook are generated directly from the dataset loaded in Section 8.

## 23. Limitations

- Empirical (historical) probabilities describe the past; they are **not predictive guarantees** for future matches — no machine-learning model was trained in this project.
- Team-name standardization (Section 10.1) only covers commonly known franchise renames; any name not in the mapping dictionary is treated as a separate team.
- Venue name cleaning only trims whitespace — near-duplicate venue names referring to the same ground are not fuzzy-merged, which may slightly split some venues' statistics.
- The team-vs-team head-to-head matrix (Section 19.5) reflects `team1`/`team2` assignment in the raw data, not confirmed home/away status.
- Strike rate and economy calculations assume every `ball` row in `deliveries.csv` represents one legal or recorded delivery; if the dataset separately flags wides/no-balls in a way that affects ball-counting, this should be re-checked against the actual column definitions before quoting exact strike rates in a viva.
- Correlation shown in Section 12.6 does not imply causation.

## 24. Future Scope

- Build an actual supervised machine-learning model (e.g. logistic regression) to predict match outcome from pre-match features, clearly separated from and compared against the empirical probabilities in this project.
- Incorporate powerplay (overs 1–6) and death-over (overs 16–20) scoring analysis if a reliable `over` column is confirmed to support it.
- Fuzzy-match near-duplicate venue names for a more precise venue-level analysis.
- Extend the dashboard to an interactive tool (e.g. using ipywidgets or Plotly) allowing season/team filtering.
- Add a Seaborn pairplot for a curated subset of numeric player metrics as an exploratory extension.